In [1]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI

# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [2]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [3]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *
from utils.file_ops import *

In [4]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel.tail(5)

,add_id,app_notes,good_match,app_status,outcome,network email,when?,institution,location,add_rank,...,add_posting,add_how_apply,other_link,how_to_rec_letters,need_cover_letter,need_RS,need_TS,OMER_letter_status,KLAUS_letter_status,WOO_letter_status
125,carleton,NaN,NaN,generating docs,NaN,NaN,NaT,Carleton College,NaN,assistant,...,JOE,NaN,NaN,NaN,1,0,1,NaN,NaN,NaN
126,leibniz,NaN,NaN,generating docs,NaN,NaN,NaT,Leibniz Centre for European Economic Research,NaN,postdoc,...,https://jobs.zew.de/jobposting/a3948eae149a2cb...,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN
127,norway_bank,NaN,NaN,generating docs,NaN,NaN,NaT,Bank of Norway,NaN,NaN,...,EJM,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN
128,carnie_melon_1,NaN,NaN,generating docs,NaN,NaN,NaT,Carnegie Mellon University,NaN,assistant,...,https://apply.interfolio.com/175370,NaN,NaN,NaN,1,1,1,NaN,NaN,NaN
129,columbia,NaN,NaN,completed,NaN,NaN,NaT,Columbia University,NaN,assistant,...,EJM,https://apply.interfolio.com/175578,NaN,NaN,0,0,0,NaN,NaN,NaN


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [5]:
# Research Statement template
with open(path_templates / "base_text/research_statement.txt", "r") as f:
    lines = f.readlines()

RA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/teaching_statement.txt", "r") as f:
    lines = f.readlines()

TA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/cover_letter.txt", "r") as f:
    lines = f.readlines()

CL_TEMPLATE = " ".join(lines)

# Figure out what docs are needed to generate

In [6]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "need_cover_letter",
    "need_RS",
    "need_TS",
]

In [7]:
completed_docs = [str(d).split("/")[-1] for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- aalto_postdoc
- berlinse
- norway_bank
- carleton
- cat_lisbon
- colorado_col
- colegio_mx
- ifo
- andersen
- coppen
- zurich
- cal_state
- carnie_melon_2
- ucl_2
- bank_spain
- lmu
- colby
- creighton
- leibniz
- puc_rio
- carnie_melon_1
- columbia
- chrisnew


In [8]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS
14,chrisnew,Christopher Newport University,NaN,assistant,NaN,research,1,0,1
28,andersen,Andersen Institute for Finance and Economics,NaN,NaN,NaN,research,1,0,0
40,colorado_col,Colorado College,NaN,assistant,NaN,research,1,1,1
41,creighton,Creighton University,NaN,assistant,NaN,teaching,1,0,0
64,colby,Colby College,NaN,assistant,NaN,research,1,0,1


# Add the context and additional data for each prompt

For these all the `raw` documents are transcribed in a few bullet points:

- For Job add use 
    ```
    summarize the job add in a few bullet points. Prioritize the attributes, qualitites they look for in a candidate
    ```
- For the isntitutional values use 
    ```
    summarize the institution mission and values in a few bullet points
    ```
- For econ research department use 
    ```
    summarize the economics department or center values and research topics in a few bullet points. Make emphasis on potential co-authors and researcg/teaching groups described there and write it in the file
    ```

In [9]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START JOB DESCRIPTION",
        end_marker="END JOB DESCRIPTION",
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START ECON DEPT",
        end_marker="END ECON DEPT",
    ),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START INSTITUTIONAL DESCRIPTION",
        end_marker="END INSTITUTIONAL DESCRIPTION",
    ),
    axis=1,
)

# OpenAI Batches

In [10]:
openai_client = OpenAI()
model_name = "gpt-4.1-mini"

In [11]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

In-process batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
52,2025-10-22 20:13:16.642377,Generate Cover Letters,gpt-4.1-mini,cl,batch_68f9812dfd58819089a9a4f33ee6db3c,completed,False,file-K7Jnca2wgmPH5WU2Nnbfum,file-CaCD8uCRY1W9u4UrcYXc5V,NaN
53,2025-10-22 20:13:21.535827,Generate Research Statements,gpt-4.1-mini,rs,batch_68f98131e3fc8190aa42395774cd4d65,completed,False,file-UjwKhii6aFUmDd7QJo6UTL,file-7Mx4XCSa2vdk2acS6zBMNE,NaN


In [12]:
# batch_history

In [13]:
class BodyText(BaseModel):
    body_text: str
    cot: str
    model_config = ConfigDict(extra="forbid")

# Cover Letter

In [14]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()
# gen_aux

In [15]:
def cl_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to add a position fit paragraph to the following COVER LETTER START and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    Keep the length of the paragraph to about 100 words.
    Use a professional and academic tone, but make it engaging and easy to read.
    For each statement of fit add an EXAMPLE from my research, teaching, or service experience that demostrates the fit statement.
    For example, for interdisciplinary research you can add that I won Dedman College Interdisciplinary Institute’s Inaugural Graduate Student Summer Research and Writing Fellowship.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full new paragraph as `body_text` and a short chain-of-thought explanation of how you modified the BASE COVER LETTER START to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE COVER LETTER START.

    BASE COVER LETTER START:
    {CL_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [16]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: cl_prompt(row),
    axis=1,
)

In [17]:
name_jsonl = "gen_cl"
description_batch = "Generate Cover Letters"

content_schema = BodyText
cat_type = "cl"

In [18]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and CL
Updating the existing old_docs
No CL to generate using gpt-4.1-mini


# Research Statement

In [19]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()
# gen_aux

In [20]:
def rs_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE RESEARCH STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my research fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the research statement to about 600 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full research statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE RESEARCH STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE RESEARCH STATEMENT:
    {RA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [21]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: rs_prompt(row),
    axis=1,
)

In [22]:
name_jsonl = "gen_rs"
description_batch = "Generate Research Statements"

content_schema = BodyText
cat_type = "rs"

In [23]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and RS
Updating the existing old_docs
No RS to generate using gpt-4.1-mini


# Teaching Statement

In [24]:
gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()
# gen_aux

In [25]:
def ts_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE TEACHING STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my educational philosophy fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the teaching statement to about 300 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full teaching statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE TEACHING STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE TEACHING STATEMENT:
    {TA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [26]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: ts_prompt(row),
    axis=1,
)

In [27]:
name_jsonl = "gen_ts"
description_batch = "Generate Teaching Statements"

content_schema = BodyText
cat_type = "ts"

In [28]:
%run -i batched_chat_gpt.py

No TS to generate using gpt-4.1-mini


# Generate the Docs

Here we assume that all docs have been generated

In [29]:
def generate_docs(df, generated_df, gen_type="rs"):
    if gen_type == "rs":
        dir_name = "research_statement"
        file_name = "Gonzalez_RS.typ"
    elif gen_type == "ts":
        dir_name = "teaching_statement"
        file_name = "Gonzalez_TS.typ"
    elif gen_type == "cl":
        dir_name = "cover_letter"
        file_name = "Gonzalez_CL.typ"
    else:
        raise ValueError("gen_type not recognized")

    for i, row in df.iterrows():
        print(f"Processing {row.add_id}...")

        new_content = generated_df[
            generated_df.add_id == row.add_id
        ].body_text_1.values[0]

        target_path = path_output / row.add_id
        subdir_path = target_path / dir_name

        if not target_path.exists():
            print(f"Creating directory: {target_path}")
            copy_and_rename_dir(path_templates / f"{dir_name}", target_path, dir_name)
        else:
            print(f"Main Directory already exists: {target_path}")
            if not subdir_path.exists():
                print(f"Creating subdirectory: {subdir_path}")
                copy_and_rename_dir(
                    path_templates / f"{dir_name}", target_path, dir_name
                )

        try:
            add_lines_to_typs_doc(subdir_path / file_name, new_content)
            print(f"{file_name} added for {row.add_id}")
        except Exception as e:
            print(f"Error processing {row.add_id}: {e}")

## Cover Letter

In [30]:
cl_gen_df = pd.read_csv(path_output / "cl_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()

generate_docs(gen_aux, cl_gen_df, gen_type="cl")

Processing chrisnew...
Creating directory: ../output_docs/chrisnew
Gonzalez_CL.typ added for chrisnew
Processing andersen...
Creating directory: ../output_docs/andersen
Gonzalez_CL.typ added for andersen
Processing colorado_col...
Creating directory: ../output_docs/colorado_col
Gonzalez_CL.typ added for colorado_col
Processing creighton...
Creating directory: ../output_docs/creighton
Gonzalez_CL.typ added for creighton
Processing colby...
Creating directory: ../output_docs/colby
Gonzalez_CL.typ added for colby
Processing bank_spain...
Creating directory: ../output_docs/bank_spain
Gonzalez_CL.typ added for bank_spain
Processing puc_rio...
Creating directory: ../output_docs/puc_rio
Gonzalez_CL.typ added for puc_rio
Processing zurich...
Creating directory: ../output_docs/zurich
Gonzalez_CL.typ added for zurich
Processing berlinse...
Creating directory: ../output_docs/berlinse
Gonzalez_CL.typ added for berlinse
Processing cal_state...
Creating directory: ../output_docs/cal_state
Gonzalez_C

## Research Statement

In [31]:
rs_gen_df = pd.read_csv(path_output / "rs_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()

generate_docs(gen_aux, rs_gen_df, gen_type="rs")

Processing colorado_col...
Main Directory already exists: ../output_docs/colorado_col
Creating subdirectory: ../output_docs/colorado_col/research_statement
Gonzalez_RS.typ added for colorado_col
Processing berlinse...
Main Directory already exists: ../output_docs/berlinse
Creating subdirectory: ../output_docs/berlinse/research_statement
Gonzalez_RS.typ added for berlinse
Processing carnie_melon_2...
Main Directory already exists: ../output_docs/carnie_melon_2
Creating subdirectory: ../output_docs/carnie_melon_2/research_statement
Gonzalez_RS.typ added for carnie_melon_2
Processing coppen...
Main Directory already exists: ../output_docs/coppen
Creating subdirectory: ../output_docs/coppen/research_statement
Gonzalez_RS.typ added for coppen
Processing carnie_melon_1...
Main Directory already exists: ../output_docs/carnie_melon_1
Creating subdirectory: ../output_docs/carnie_melon_1/research_statement
Gonzalez_RS.typ added for carnie_melon_1


## Teaching Statement

In [32]:
rt_gen_df = pd.read_csv(path_output / "ts_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()

generate_docs(gen_aux, rt_gen_df, gen_type="ts")

Processing chrisnew...
Main Directory already exists: ../output_docs/chrisnew
Creating subdirectory: ../output_docs/chrisnew/teaching_statement
Gonzalez_TS.typ added for chrisnew
Processing colorado_col...
Main Directory already exists: ../output_docs/colorado_col
Creating subdirectory: ../output_docs/colorado_col/teaching_statement
Gonzalez_TS.typ added for colorado_col
Processing colby...
Main Directory already exists: ../output_docs/colby
Creating subdirectory: ../output_docs/colby/teaching_statement
Gonzalez_TS.typ added for colby
Processing cal_state...
Main Directory already exists: ../output_docs/cal_state
Creating subdirectory: ../output_docs/cal_state/teaching_statement
Gonzalez_TS.typ added for cal_state
Processing carnie_melon_2...
Main Directory already exists: ../output_docs/carnie_melon_2
Creating subdirectory: ../output_docs/carnie_melon_2/teaching_statement
Gonzalez_TS.typ added for carnie_melon_2
Processing carleton...
Main Directory already exists: ../output_docs/carl